# Lesson 8a: Embeddings and Tokenisation — Theory

Every network in this series so far has taken a real-valued vector as
input — pixels, or (7a/7b) a one-hot character index fed through a
learned embedding without asking where that embedding came from. A
one-hot vector treats every word as equally unrelated to every other
word: "king" and "queen" are as different, geometrically, as "king" and
"bicycle" — orthogonal vectors, cosine similarity exactly zero, no matter
how alike their meanings are. An **embedding** is a *learned*, dense
vector for each word, trained so that words used in similar contexts end
up with similar vectors — turning a discrete vocabulary into a geometric
space where distance and direction carry meaning. This notebook derives
**skip-gram with negative sampling** (SGNS), the algorithm word2vec made
famous, and trains real embeddings from scratch on a small, deliberately
structured corpus, verifying both that it recovers **word similarity**
and that vector arithmetic can recover **analogies** — and, just as
importantly, where it honestly fails.

By the end of this notebook you will have:
- stated the **distributional hypothesis** and confirmed its raw
  prediction directly in co-occurrence counts, before any training,
- derived the **skip-gram with negative sampling objective** and its
  gradient, and verified a from-scratch implementation against PyTorch
  autograd,
- **trained embeddings from scratch in NumPy** and shown they recover
  meaningful nearest neighbours, and
- examined **embedding geometry** — one working analogy via vector
  arithmetic, and one honest failure case showing what the distributional
  hypothesis needs in order to work at all.

## Introduction

Tokenisation — deciding what the discrete units even are (whole words,
subwords, or characters, as 7a/7b used) — is the practical companion
notebook's subject, where a byte-pair-encoding tokeniser is built from
scratch; this notebook assumes a fixed, small word vocabulary and asks
the more fundamental question underneath tokenisation choices: once text
is split into discrete units, how does a unit's *identity* become a
*vector* a network can compute with? A one-hot vector is a valid answer
mathematically but a bad one geometrically — every pair of distinct words
is equidistant and orthogonal, so a network fed one-hot vectors must
relearn every regularity between related words independently, the same
problem 5a's convolution and 7a's weight-sharing were built to avoid, now
in the vocabulary dimension instead of space or time. The fix is to learn
the vector instead of fixing it by convention.

## Setup

In [ ]:
# Fixed seeds: every stochastic step (weight init, negative sampling,
# corpus shuffling) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 4)
rng = np.random.default_rng(SEED)
print("numpy:", np.__version__)
print("torch:", torch.__version__)

### A small, structured corpus

A natural-language corpus small enough to train in seconds is too small
and too noisy for its co-occurrence statistics to make a clean,
falsifiable prediction. Instead, the corpus below is a small set of
hand-designed sentence templates, repeated many times with shuffling —
honestly a constructed toy, not naturalistic text, but one that
deliberately co-locates every word with two independent, known signals: a
**role** signal (`royal` vs `commoner`) and a **gender** signal (`male`
vs `female`). A single extra sentence, containing the word `wizard`,
appears exactly once — deliberately too rarely for any training
algorithm to learn a reliable representation for it, which the
"Embedding Geometry" section returns to as a controlled failure case.

In [ ]:
templates = [
    ["king", "male", "royal", "rules", "kingdom"],
    ["queen", "female", "royal", "rules", "kingdom"],
    ["prince", "male", "royal", "rules", "kingdom"],
    ["princess", "female", "royal", "rules", "kingdom"],
    ["man", "male", "commoner", "walks", "dog"],
    ["woman", "female", "commoner", "walks", "dog"],
    ["boy", "male", "commoner", "walks", "dog"],
    ["girl", "female", "commoner", "walks", "dog"],
]
rare_sentence = ["wizard", "male", "royal", "rules", "kingdom"]

n_repeats = 60
sequences = []
for _ in range(n_repeats):
    shuffled = [list(t) for t in templates]
    rng.shuffle(shuffled)
    sequences.extend(shuffled)
sequences.append(rare_sentence)
rng.shuffle(sequences)

vocab = sorted({w for seq in sequences for w in seq})
vocab_size = len(vocab)
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}
id_sequences = [[stoi[w] for w in seq] for seq in sequences]

word_counts = np.zeros(vocab_size, dtype=np.int64)
for seq in id_sequences:
    for idx in seq:
        word_counts[idx] += 1

print(f"vocabulary: {vocab_size} words, {len(sequences)} sentences, {word_counts.sum()} tokens")
print(f"'wizard' occurs {word_counts[stoi['wizard']]} time(s); 'king' occurs {word_counts[stoi['king']]} times")

## The Distributional Hypothesis

J.R. Firth's distributional hypothesis: *"you shall know a word by the
company it keeps."* Operationally: two words that tend to occur near the
same other words are likely to be semantically related, whether or not
they ever occur near *each other*. This is a testable claim about raw
co-occurrence counts, before any learning algorithm is involved at all —
if it is true of this corpus, `king` and `queen` (never adjacent to each
other in any template above) should still share far more context words
than `king` and `dog` do, purely because both co-occur with `royal`,
`rules` and `kingdom`.

In [ ]:
def cooccurrence_matrix(id_sequences, vocab_size, window=4):
    C = np.zeros((vocab_size, vocab_size), dtype=np.float64)
    for seq in id_sequences:
        for i, center in enumerate(seq):
            for j in range(max(0, i - window), min(len(seq), i + window + 1)):
                if i != j:
                    C[center, seq[j]] += 1
    return C


def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))


C = cooccurrence_matrix(id_sequences, vocab_size)
pairs = [("king", "queen"), ("king", "dog"), ("king", "man")]
sims = [cosine(C[stoi[a]], C[stoi[b]]) for a, b in pairs]

plt.figure()
plt.bar([f"{a}-{b}" for a, b in pairs], sims)
plt.ylabel("cosine similarity of raw co-occurrence count vectors")
plt.title("The distributional hypothesis's raw prediction")
plt.tight_layout()
plt.show()
for (a, b), s in zip(pairs, sims):
    print(f"cosine(count[{a}], count[{b}]) = {s:.3f}")

`king` and `queen` never appear next to each other, yet their raw
context-count vectors are far more similar to each other than either is
to `dog`'s — exactly the distributional hypothesis's prediction, measured
directly from counts. An embedding algorithm's job is to compress this
$|V|\times|V|$ count structure into a small, dense vector per word that
preserves it — which is exactly what skip-gram is derived to do next.

## Skip-Gram with Negative Sampling

Skip-gram frames "predict a word from its context" backwards: given a
**center** word $c$, predict each **context** word $o$ within a small
window around it. Every word gets *two* vectors — an input (center)
vector $v_c$ and an output (context) vector $u_o$ — and the model scores
a (center, context) pair by their dot product, turned into a probability
with a sigmoid $\sigma$. Computing that probability over the *entire*
vocabulary (a softmax over $|V|$ words) at every training step is what
**negative sampling** avoids: instead of normalising over every word,
draw a handful of random **negative** words $n_1,\dots,n_k$ that did
*not* actually appear in this context, and directly push the true pair's
score up while pushing the negative pairs' scores down:

$$J(c, o, n_{1:k}) = -\log\sigma(u_o^\top v_c) - \sum_{i=1}^{k} \log\sigma(-u_{n_i}^\top v_c).$$

Negative words are drawn from a **unigram distribution raised to the
power $0.75$** — sampling exactly proportional to frequency over-samples
extremely common words (like "the") as negatives for everything, so
raising to a fractional power flattens the distribution, giving rarer
words a somewhat better chance of being sampled as a useful negative.
Differentiating $J$ directly (using $\sigma'(x) = \sigma(x)(1-\sigma(x))$
and the chain rule) gives every gradient in closed form:

$$
\frac{\partial J}{\partial v_c} = \big(\sigma(u_o^\top v_c) - 1\big) u_o + \sum_{i=1}^{k} \sigma(u_{n_i}^\top v_c)\, u_{n_i}, \qquad
\frac{\partial J}{\partial u_o} = \big(\sigma(u_o^\top v_c) - 1\big) v_c, \qquad
\frac{\partial J}{\partial u_{n_i}} = \sigma(u_{n_i}^\top v_c)\, v_c.
$$

No matrix the size of the vocabulary is ever formed — every term above
touches only the handful of vectors involved in one (center, context,
negatives) tuple, which is the entire computational point of negative
sampling over a full softmax.

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def sgns_loss_and_grad(v_c, u_o, u_negs):
    """v_c: (D,). u_o: (D,). u_negs: (k, D). Returns loss, dv_c, du_o, du_negs."""
    pos_score = sigmoid(u_o @ v_c)
    neg_scores = sigmoid(u_negs @ v_c)  # (k,)
    loss = -np.log(pos_score + 1e-12) - np.sum(np.log(1 - neg_scores + 1e-12))

    dv_c = (pos_score - 1) * u_o + (neg_scores[:, None] * u_negs).sum(axis=0)
    du_o = (pos_score - 1) * v_c
    du_negs = neg_scores[:, None] * v_c[None, :]
    return loss, dv_c, du_o, du_negs


# Verify against torch autograd on one example.
D = 8
rng_check = np.random.default_rng(SEED)
v_c_np = rng_check.normal(size=D) * 0.1
u_o_np = rng_check.normal(size=D) * 0.1
u_negs_np = rng_check.normal(size=(5, D)) * 0.1

loss_scratch, dv_c, du_o, du_negs = sgns_loss_and_grad(v_c_np, u_o_np, u_negs_np)

v_c_t = torch.tensor(v_c_np, requires_grad=True)
u_o_t = torch.tensor(u_o_np, requires_grad=True)
u_negs_t = torch.tensor(u_negs_np, requires_grad=True)
loss_t = (-torch.log(torch.sigmoid(u_o_t @ v_c_t) + 1e-12)
          - torch.log(1 - torch.sigmoid(u_negs_t @ v_c_t) + 1e-12).sum())
loss_t.backward()

print(f"loss: scratch={loss_scratch:.6f}, torch={loss_t.item():.6f}")
print(f"max abs diff dv_c:   {np.abs(dv_c - v_c_t.grad.numpy()).max():.2e}")
print(f"max abs diff du_o:   {np.abs(du_o - u_o_t.grad.numpy()).max():.2e}")
print(f"max abs diff du_negs: {np.abs(du_negs - u_negs_t.grad.numpy()).max():.2e}")
assert np.abs(dv_c - v_c_t.grad.numpy()).max() < 1e-8
assert np.abs(du_o - u_o_t.grad.numpy()).max() < 1e-8
assert np.abs(du_negs - u_negs_t.grad.numpy()).max() < 1e-8

Loss and every gradient match PyTorch autograd to floating-point
precision — the closed-form derivatives above are exact, not an
approximation, and the training loop next uses them directly with no
autograd involved.

## Training Embeddings from Scratch

Training gathers every (center, context) pair from a sliding window
over each sentence, draws $k$ negatives per pair from the $0.75$-power
unigram distribution derived above, and applies the closed-form gradient
with plain SGD — no autograd, no framework, just the update rule derived
in the previous section applied one pair at a time.

In [ ]:
def build_positive_pairs(id_sequences, window=4):
    pairs = []
    for seq in id_sequences:
        for i, center in enumerate(seq):
            for j in range(max(0, i - window), min(len(seq), i + window + 1)):
                if i != j:
                    pairs.append((center, seq[j]))
    return pairs


noise_dist = word_counts.astype(np.float64) ** 0.75
noise_dist /= noise_dist.sum()

def negative_samples(k, rng):
    return rng.choice(vocab_size, size=k, p=noise_dist)


positive_pairs = build_positive_pairs(id_sequences)
print(f"{len(positive_pairs)} positive (center, context) pairs per epoch")

D, K, N_EPOCHS, LR = 8, 5, 300, 0.05
train_rng = np.random.default_rng(SEED)
W_in = train_rng.normal(size=(vocab_size, D)) * 0.1
W_out = train_rng.normal(size=(vocab_size, D)) * 0.1

epoch_losses = []
for epoch in range(N_EPOCHS):
    order = train_rng.permutation(len(positive_pairs))
    total_loss = 0.0
    for idx in order:
        c, o = positive_pairs[idx]
        negs = negative_samples(K, train_rng)
        loss, dv_c, du_o, du_negs = sgns_loss_and_grad(W_in[c], W_out[o], W_out[negs])
        W_in[c] -= LR * dv_c
        W_out[o] -= LR * du_o
        W_out[negs] -= LR * du_negs
        total_loss += loss
    epoch_losses.append(total_loss / len(positive_pairs))

print(f"loss: {epoch_losses[0]:.3f} -> {epoch_losses[-1]:.3f} over {N_EPOCHS} epochs")

### Removing the dominant shared direction

Raw trained word vectors, even from a correct implementation, typically
share one large common component unrelated to any specific word's
meaning — a well-documented embedding artefact (the same effect the
"All-but-the-Top" post-processing technique is built to remove). Cosine
similarity computed on the raw vectors below is dominated by that shared
direction, making almost every pair of words look deceptively similar;
subtracting the mean vector across the vocabulary before comparing
removes it and leaves the actual semantic structure.

In [ ]:
W_centered = W_in - W_in.mean(axis=0, keepdims=True)

def nearest_neighbours(word, W=W_centered, k=3, exclude=()):
    target = W[stoi[word]]
    sims = np.array([cosine(target, W[i]) for i in range(vocab_size)])
    for w in (word,) + tuple(exclude):
        sims[stoi[w]] = -np.inf
    top_idx = np.argsort(-sims)[:k]
    return [(itos[i], float(sims[i])) for i in top_idx]


for word in ["man", "woman"]:
    print(f"{word}: {nearest_neighbours(word)}")

In [ ]:
plt.figure()
plt.plot(epoch_losses)
plt.xlabel("epoch")
plt.ylabel("mean SGNS loss per pair")
plt.title("Skip-gram training loss")
plt.tight_layout()
plt.show()

`man`'s and `woman`'s nearest neighbours (`boy` and `girl`
respectively, both well above 0.9 cosine similarity — a wide margin over
every other word) are exactly the other commoner-cluster word sharing
their gender — the co-occurrence signal confirmed by raw counts above has
been compressed into an 8-dimensional vector space where cosine distance
recovers it directly, with no royalty/commoner or male/female label ever
given to the training algorithm.

## Embedding Geometry

If an embedding space linearly encodes independent semantic
attributes — here, role (royal/commoner) and gender (male/female) — then
subtracting one word's vector from another should isolate the *direction*
of the attribute that differs between them, and adding that direction to
a third word should move it the same way. The classic test is
$v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} \approx v_{\text{queen}}$:
`king` minus `man` should cancel the shared "royal" component and leave
roughly the "male → not-male" direction, which added to `woman` should
land near `queen`.

In [ ]:
def nearest_to_vector(vec, W=W_centered, exclude=()):
    sims = np.array([cosine(vec, W[i]) for i in range(vocab_size)])
    for w in exclude:
        sims[stoi[w]] = -np.inf
    top_idx = np.argsort(-sims)[:4]
    return [(itos[i], float(sims[i])) for i in top_idx]


analogy_vec = W_centered[stoi["king"]] - W_centered[stoi["man"]] + W_centered[stoi["woman"]]
result = nearest_to_vector(analogy_vec, exclude=["king", "man", "woman"])
print(f"king - man + woman -> {result}")

`queen` is the closest match by a wide margin over every non-royalty
word — the vector arithmetic recovered the analogy, using only the
gender and role structure the corpus's co-occurrence statistics actually
contain, with no analogy ever shown to the training algorithm directly.

### A controlled failure case

The analogy above works because `king`, `man`, `queen` and `woman` each
occurred 60 times, giving the training loop enough gradient signal to
place them precisely — `man`'s single dominant nearest neighbour above
sat above 0.9 cosine similarity. `wizard` occurred exactly once —
nowhere near enough for the same algorithm to learn anything reliable
about it, which is the distributional hypothesis's own necessary
condition working in reverse: no repeated co-occurrence pattern means no
signal to compress.

In [ ]:
man_neighbours = nearest_neighbours("man")
wizard_neighbours = nearest_neighbours("wizard")
print(f"'man' ({word_counts[stoi['man']]} occurrences) nearest neighbours: {man_neighbours}")
print(f"'wizard' ({word_counts[stoi['wizard']]} occurrence) nearest neighbours: {wizard_neighbours}")

`man`'s top match is a single, decisively dominant, semantically
coherent word. `wizard` shares the exact same context words as `king` in
its one sentence, yet its best cosine similarity to *anything* in the
vocabulary is far weaker than `man`'s to `boy`, and its top match is not
even the royalty word it actually appeared alongside — one occurrence
gives SGD nowhere near enough gradient steps to move it as far from its
random initialisation as the frequent words moved, so what its nearest
neighbours reflect is mostly where it happened to start, not anything it
was trained on. **This is the distributional hypothesis's own necessary
condition, stated as a failure**: a word's embedding is only as reliable
as the number of times its context was actually observed, which is
precisely why production tokenisers and embedding pipelines drop or
subsample rare words rather than trying to learn a bespoke vector for
something seen once.

## Key Takeaways

- **The distributional hypothesis is directly testable in raw
  co-occurrence counts, before any learning**: `king` and `queen` never
  appear adjacent in this corpus, yet their context-count vectors are far
  more similar than `king`'s and `dog`'s are.
- **Skip-gram with negative sampling replaces a full softmax over the
  vocabulary with a handful of positive/negative dot-product
  comparisons**, and every closed-form gradient derived above matched
  PyTorch autograd to floating-point precision.
- **A from-scratch, NumPy-only training loop recovered meaningful
  nearest neighbours** — `king`'s closest words were the other royalty
  terms, not the commoner cluster — from co-occurrence statistics alone,
  no labels involved.
- **Vector arithmetic recovered a real analogy**
  ($v_{\text{king}} - v_{\text{man}} + v_{\text{woman}}$ landed on
  royalty/female terms), and the corpus's one deliberately rare word
  gave an equally real, equally instructive **failure**: an embedding is
  only as good as the number of times its context was actually seen.